# LLM Training Lab, lab 6. Generation and the KV cache, implemented

INFERENCE & EVAL. No weights change in the lab steps. You pretrain a small GPT on a CPU first, then sample from it, add a KV cache with a RoPE position offset, prove the cached logits match, count the cache bytes and time it.

Runs on a CPU. Steps follow the lab page in this app; run the cells in order.

In [ ]:
%pip install -q "transformers==5.22.1" "tokenizers==0.23.3" "huggingface_hub==2.4.0"


If the install cell changed a package that was already loaded, use Runtime > Restart session, then run the check below.

In [ ]:
import importlib, platform, shutil, sys

# Runtime check. Every line that starts with FIX names the change to make before you go on.
PINS = {"transformers": "5.22.1", "tokenizers": "0.23.3", "huggingface_hub": "2.4.0"}
OURS = {"numpy": "2.5.4"}  # not installed here: torch is built against the runtime's own numpy
print("Python", platform.python_version())
if sys.version_info < (3, 11):
    print("FIX  Python 3.11 or later is needed, and 3.12 is recommended. Colab's default runtime has it.")
for name in ["torch", "numpy", "tokenizers", "huggingface_hub", "transformers"]:
    try:
        v = importlib.import_module(name).__version__
        if name in OURS:
            note = f"   (our CPU runs used {OURS[name]})"
        elif name in PINS and v != PINS[name]:
            note = f"   FIX  run the install cell above, then Runtime > Restart session (pinned {PINS[name]})"
        else:
            note = ""
        print(f"{name:16} {v}{note}")
    except ImportError:
        print(f"FIX  {name} is missing. Run the install cell above.")
free = shutil.disk_usage(".").free / 1e9
print(f"free disk {free:.1f} GB")
if free < 2:
    print("FIX  under 2 GB free. Step 6 downloads SmolLM2-135M (about 270 MB) and the data file is 22 MB.")
try:
    import psutil
    print(f"RAM {psutil.virtual_memory().total / 1e9:.1f} GB")
except ImportError:
    pass
import torch
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. This lab runs on a CPU, so none is needed.")


## Setup. The model

The GPT from lltGPT, with RMSNorm, RoPE, SwiGLU and tied embeddings, plus `n_kv_heads` for step 8.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The GPT from lltGPT and lltTinyStories: pre-norm RMSNorm, RoPE, SwiGLU, tied embeddings.
# One addition: n_kv_heads, for grouped-query attention in step 8 (equal to n_heads by default).

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)

def rope_tables(start, T, hd):
    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2).float() / hd))
    f = torch.outer(torch.arange(start, start + T).float(), inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2).type_as(x)

class Block(nn.Module):
    def __init__(self, d, h, h_kv):
        super().__init__()
        self.h, self.h_kv, self.hd = h, h_kv, d // h
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.qkv = nn.Linear(d, d + 2 * h_kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        hid = int(8 * d / 3 / 64 + 0.999) * 64
        self.w1 = nn.Linear(d, hid, bias=False)
        self.w3 = nn.Linear(d, hid, bias=False)
        self.w2 = nn.Linear(hid, d, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(self.n1(x)).split([C, self.h_kv * self.hd, self.h_kv * self.hd], dim=2)
        q = q.view(B, T, self.h, self.hd).transpose(1, 2)
        k = k.view(B, T, self.h_kv, self.hd).transpose(1, 2)
        v = v.view(B, T, self.h_kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.h // self.h_kv, dim=1)
        v = v.repeat_interleave(self.h // self.h_kv, dim=1)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))
        h = self.n2(x)
        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))

class GPT(nn.Module):
    def __init__(self, V, d, L, H, ctx, H_kv=None):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.blocks = nn.ModuleList([Block(d, H, H_kv or H) for _ in range(L)])
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.emb.weight
        self.H, self.ctx = H, ctx
        for n, p in self.named_parameters():
            if p.dim() == 2:
                std = 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02
                nn.init.normal_(p, 0.0, std)
    def forward(self, idx):
        cos, sin = rope_tables(0, idx.shape[1], self.emb.embedding_dim // self.H)
        x = self.emb(idx)
        for b in self.blocks:
            x = b(x, cos, sin)
        return self.head(self.norm(x))


## Setup. Pretrain the lab model on a CPU

2.2M parameters, 1,500 steps of 16 x 256 tokens on TinyStoriesV2-GPT4-valid.txt (CDLA-Sharing-1.0). Our run lltGeneration-t1-model-s0 went from val loss 7.67 at step 0 to 2.16 at step 1,500.

In [ ]:
import math, random, time
import numpy as np
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

# Pretrain the lab model FROM SCRATCH on CPU (same recipe as train_small.py, run lltGeneration-t1-model-s0).
torch.set_num_threads(4)
path = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset")
stories = [s.strip() for s in open(path, encoding="utf-8").read().split("<|endoftext|>") if s.strip()]
random.Random(0).shuffle(stories)
cut = int(0.95 * len(stories))
train_text, val_text = stories[:cut], stories[cut:]

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator(train_text, trainers.BpeTrainer(vocab_size=2048, special_tokens=["<|eot|>"],
                        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
EOT = tok.token_to_id("<|eot|>")
def encode(texts):
    ids = []
    for enc in tok.encode_batch(texts):
        ids.extend(enc.ids)
        ids.append(EOT)
    return np.array(ids, dtype=np.int64)
tr, va = encode(train_text), encode(val_text)

torch.manual_seed(0)
rng = np.random.default_rng(0)
model = GPT(2048, 192, 4, 6, 256)
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
                        lr=2e-3, betas=(0.9, 0.95))
STEPS, T, B = 1500, 256, 16
vx = torch.tensor(np.stack([va[s:s + T] for s in range(0, 64 * T, T)]))
vy = torch.tensor(np.stack([va[s + 1:s + T + 1] for s in range(0, 64 * T, T)]))
def lr_at(s):
    if s < 100:
        return 2e-3 * (s + 1) / 100
    return 2e-3 * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - 100) / (STEPS - 100))))
t0 = time.time()
for step in range(STEPS + 1):
    if step % 250 == 0:
        with torch.no_grad():
            model.eval()
            vl = sum(F.cross_entropy(model(vx[i:i + 16]).view(-1, 2048), vy[i:i + 16].reshape(-1), reduction="sum").item()
                     for i in range(0, 64, 16)) / vy.numel()
            model.train()
        print(f"step {step:5d}  val loss {vl:.4f}  {time.time() - t0:.0f} s")
    if step == STEPS:
        break
    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    ix = rng.integers(0, tr.size - T - 1, B)
    x = torch.tensor(np.stack([tr[i:i + T] for i in ix]))
    y = torch.tensor(np.stack([tr[i + 1:i + T + 1] for i in ix]))
    loss = F.cross_entropy(model(x).view(-1, 2048), y.view(-1))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
model.eval()
print("parameters", sum(p.numel() for p in model.parameters()))


## Step 1. Temperature, top-k and top-p on eight words

The same cell runs in the lab page with NumPy.

In [ ]:
import numpy as np

words = ["mat", "sofa", "floor", "bed", "roof", "moon", "table", "piano"]
logits = np.array([3.2, 2.5, 2.1, 1.6, 0.4, -0.8, 1.1, -1.5])

def softmax(z):
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()

def temperature(z, t):
    return z / t

def top_k(z, k):
    kth = np.sort(z)[-k]
    return np.where(z >= kth, z, -np.inf)

def top_p(z, p):
    order = np.argsort(-z)
    probs = softmax(z)[order]
    before = np.cumsum(probs) - probs        # mass of the tokens ranked above each one
    keep = np.zeros(len(z), dtype=bool)
    keep[order[before < p]] = True
    return np.where(keep, z, -np.inf)

def show(name, z):
    p = softmax(z)
    kept = [f"{w} {q:.3f}" for w, q in sorted(zip(words, p), key=lambda x: -x[1]) if q > 0]
    print(f"{name:14} kept {len(kept)}  " + ", ".join(kept))

show("T = 1.0", logits)
show("T = 0.5", temperature(logits, 0.5))
show("T = 2.0", temperature(logits, 2.0))
show("top-k, k = 3", top_k(logits, 3))
show("top-p, p = 0.9", top_p(logits, 0.9))

rng = np.random.default_rng(0)
draws = rng.choice(len(words), size=1000, p=softmax(top_p(logits, 0.9)))
print("1,000 draws with top-p 0.9:", {words[i]: int(c) for i, c in zip(*np.unique(draws, return_counts=True))})


## Step 2. Generate without a cache

In [ ]:
def filter_logits(logits, temperature=1.0, top_k=0, top_p=1.0):
    z = logits / temperature
    if top_k:
        kth = torch.topk(z, top_k).values[..., -1:]
        z = z.masked_fill(z < kth, float("-inf"))
    if top_p < 1.0:
        srt, order = torch.sort(z, descending=True)
        p = srt.softmax(-1)
        drop = p.cumsum(-1) - p >= top_p          # the tokens ranked above already reach top_p
        z = z.masked_fill(drop.scatter(-1, order, drop), float("-inf"))
    return z.log_softmax(-1)

@torch.no_grad()
def generate_nocache(model, ids, n, temperature=1.0, top_k=0, top_p=1.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.tensor([ids])
    for _ in range(n):
        logits = model(x)[:, -1]                  # the whole sequence again, every step
        if temperature == 0:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            nxt = torch.multinomial(filter_logits(logits, temperature, top_k, top_p).exp(), 1, generator=g)
        x = torch.cat([x, nxt], 1)
    return x[0, len(ids):].tolist()

prompt = "Once upon a time, there was a little girl named Lily. She"
ids = tok.encode(prompt).ids
print(len(ids), "prompt tokens")
print(prompt + tok.decode(generate_nocache(model, ids, 40, temperature=0.8, top_k=50)))


## Step 3. Temperature and top-p on your model

In [ ]:
with torch.no_grad():
    logits = model(torch.tensor([ids]))[0, -1]
for T in [0.5, 0.8, 1.0, 1.5]:
    lp = filter_logits(logits, T)
    p = lp.exp()
    entropy = -(p * lp).sum().item()
    kept = int((filter_logits(logits, T, top_p=0.9) > float("-inf")).sum())
    print(f"T {T}  entropy {entropy:.2f} nats  top token {p.max().item():.2f}  top-p 0.9 keeps {kept} of {len(p)}")

def repeated_4grams(toks):
    grams = [tuple(toks[i:i + 4]) for i in range(len(toks) - 3)]
    return 1 - len(set(grams)) / len(grams)

for name, kw in [("greedy", dict(temperature=0)), ("top-p 0.9", dict(top_p=0.9))]:
    out = generate_nocache(model, ids, 200, seed=0, **kw)
    print(f"{name:10} repeated 4-grams {repeated_4grams(out):.2f}   {tok.decode(out[:24])!r}")


## Step 4. Add a preallocated KV cache with a position offset

In [ ]:
class KVCache:
    def __init__(self, n_layers, batch, n_kv_heads, max_len, head_dim, dtype=torch.float32):
        shape = (n_layers, batch, n_kv_heads, max_len, head_dim)
        self.k = torch.zeros(shape, dtype=dtype)   # allocated once, filled in place
        self.v = torch.zeros(shape, dtype=dtype)
        self.pos = 0                               # slots that hold real tokens
    def nbytes(self):
        return self.k.nbytes + self.v.nbytes

def block_forward(self, x, cos, sin, cache=None, layer=0):
    B, T, C = x.shape
    q, k, v = self.qkv(self.n1(x)).split([C, self.h_kv * self.hd, self.h_kv * self.hd], dim=2)
    q = q.view(B, T, self.h, self.hd).transpose(1, 2)
    k = k.view(B, T, self.h_kv, self.hd).transpose(1, 2)
    v = v.view(B, T, self.h_kv, self.hd).transpose(1, 2)
    q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
    p = cache.pos if cache is not None else 0
    if cache is not None:
        cache.k[layer, :, :, p:p + T] = k          # write the new keys and values
        cache.v[layer, :, :, p:p + T] = v
        k, v = cache.k[layer, :, :, :p + T], cache.v[layer, :, :, :p + T]
    mask = torch.arange(p + T)[None, :] <= (p + torch.arange(T))[:, None]   # query i reads keys 0 .. p + i
    k = k.repeat_interleave(self.h // self.h_kv, dim=1)
    v = v.repeat_interleave(self.h // self.h_kv, dim=1)
    a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    x = x + self.o(a.transpose(1, 2).reshape(B, T, C))
    h = self.n2(x)
    return x + self.w2(F.silu(self.w1(h)) * self.w3(h))

def gpt_forward(self, idx, cache=None):
    T = idx.shape[1]
    start = cache.pos if cache is not None else 0   # the RoPE position offset
    cos, sin = rope_tables(start, T, self.emb.embedding_dim // self.H)
    x = self.emb(idx)
    for i, b in enumerate(self.blocks):
        x = b(x, cos, sin, cache, i)
    if cache is not None:
        cache.pos += T
    return self.head(self.norm(x))

def new_cache(self, batch, max_len, dtype=torch.float32):
    b = self.blocks[0]
    return KVCache(len(self.blocks), batch, b.h_kv, max_len, b.hd, dtype)

Block.forward, GPT.forward, GPT.new_cache = block_forward, gpt_forward, new_cache
cache = model.new_cache(1, 64)
print(model(torch.tensor([ids]), cache).shape, "cache.pos", cache.pos)


## Step 5. Prove cached and uncached logits match

In [ ]:
@torch.no_grad()
def check_cache(model, ids, n=200):
    cache = model.new_cache(1, len(ids) + n)
    steps = [model(torch.tensor([ids]), cache)[0, -1]]         # prefill, the whole prompt at once
    toks = []
    for _ in range(n - 1):
        nxt = steps[-1].argmax().view(1, 1)
        toks.append(nxt.item())
        steps.append(model(nxt, cache)[0, -1])                  # decode, one token per call
    toks.append(steps[-1].argmax().item())
    full = model(torch.tensor([ids + toks[:-1]]))[0, len(ids) - 1:]   # no cache, one pass
    cached = torch.stack(steps)
    err = (cached - full).abs().max().item()
    print(f"{n} steps  max abs error {err:.1e}  largest logit {full.abs().max().item():.1f}")
    print("allclose:", torch.allclose(cached, full, atol=1e-4, rtol=1e-4),
          "| same greedy tokens:", toks == full.argmax(-1).tolist())

check_cache(model, ids)


## Step 6. Count the cache bytes

Your model first, then SmolLM2-135M's real Hugging Face cache (downloads about 270 MB).

In [ ]:
L, H_kv, hd = len(model.blocks), model.blocks[0].h_kv, model.blocks[0].hd
print("layers", L, "| KV heads", H_kv, "| head dim", hd)
for T, dtype, nbytes in [(256, torch.float32, 4), (256, torch.float16, 2)]:
    cache = model.new_cache(1, T, dtype)
    formula = 2 * L * T * H_kv * hd * nbytes
    print(f"T {T}  {str(dtype):13}  formula {formula:,} bytes  tensors {cache.nbytes():,} bytes")


In [ ]:
from transformers import AutoModelForCausalLM

smol = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", dtype=torch.float32).eval()
c = smol.config
T = 512
x = torch.randint(0, c.vocab_size, (1, T), generator=torch.Generator().manual_seed(0))
with torch.no_grad():
    pkv = smol(x, use_cache=True).past_key_values
real = sum(layer.keys.nbytes + layer.values.nbytes for layer in pkv.layers)
hd = c.hidden_size // c.num_attention_heads
formula = 2 * c.num_hidden_layers * T * c.num_key_value_heads * hd * 4
print("layers", c.num_hidden_layers, "| KV heads", c.num_key_value_heads, "| head dim", hd)
print(f"formula {formula:,} bytes  cache tensors {real:,} bytes")


## Step 7. Time generation with and without the cache

In [ ]:
import time

@torch.no_grad()
def generate_cached(model, ids, n):
    cache = model.new_cache(1, len(ids) + n)
    logits = model(torch.tensor([ids]), cache)[:, -1]          # prefill
    out = []
    for _ in range(n):
        nxt = logits.argmax(-1, keepdim=True)
        out.append(nxt.item())
        logits = model(nxt, cache)[:, -1]                      # decode
    return out

for n in [64, 128, 256, 512, 1024]:
    t = time.perf_counter(); a = generate_nocache(model, ids, n, temperature=0); slow = time.perf_counter() - t
    t = time.perf_counter(); b = generate_cached(model, ids, n); fast = time.perf_counter() - t
    print(f"{n:5d} tokens  no cache {slow:7.2f} s  cache {fast:6.2f} s  speedup {slow / fast:5.1f}x  same tokens {a == b}")


## Step 8. Shrink the cache with grouped-query attention

In [ ]:
for h_kv in [6, 3, 2, 1]:
    torch.manual_seed(0)
    m = GPT(2048, 192, 4, 6, 256, h_kv).eval()
    cache = m.new_cache(1, 256)
    attn = m.blocks[0].qkv.weight.numel() + m.blocks[0].o.weight.numel()
    print(f"KV heads {h_kv}  group {6 // h_kv}  cache at 256 tokens {cache.nbytes():,} bytes  attention params per layer {attn:,}")
    check_cache(m, ids, n=50)


## Optional. Reproduce the two broken runs

In [ ]:
# The two broken runs from the page. Each patch breaks one line, runs the step 5 check, then restores it.

def gpt_forward_rope_bug(self, idx, cache=None):
    T = idx.shape[1]
    cos, sin = rope_tables(0, T, self.emb.embedding_dim // self.H)   # BUG, the offset cache.pos is missing
    x = self.emb(idx)
    for i, b in enumerate(self.blocks):
        x = b(x, cos, sin, cache, i)
    if cache is not None:
        cache.pos += T
    return self.head(self.norm(x))

GPT.forward = gpt_forward_rope_bug
print("RoPE offset bug")
check_cache(model, ids, n=60)
GPT.forward = gpt_forward

def block_forward_mask_bug(self, x, cos, sin, cache=None, layer=0):
    B, T, C = x.shape
    q, k, v = self.qkv(self.n1(x)).split([C, self.h_kv * self.hd, self.h_kv * self.hd], dim=2)
    q = q.view(B, T, self.h, self.hd).transpose(1, 2)
    k = k.view(B, T, self.h_kv, self.hd).transpose(1, 2)
    v = v.view(B, T, self.h_kv, self.hd).transpose(1, 2)
    q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
    p = cache.pos if cache is not None else 0
    if cache is not None:
        cache.k[layer, :, :, p:p + T] = k
        cache.v[layer, :, :, p:p + T] = v
        k, v = cache.k[layer, :, :, :p + T], cache.v[layer, :, :, :p + T]
    mask = torch.arange(p + T)[None, :] <= (p + torch.arange(T))[:, None]
    if cache is not None and T > 1:
        mask = None                                                    # BUG, the prefill chunk reads ahead
    k = k.repeat_interleave(self.h // self.h_kv, dim=1)
    v = v.repeat_interleave(self.h // self.h_kv, dim=1)
    a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    x = x + self.o(a.transpose(1, 2).reshape(B, T, C))
    h = self.n2(x)
    return x + self.w2(F.silu(self.w1(h)) * self.w3(h))

Block.forward = block_forward_mask_bug
print("prefill mask bug")
check_cache(model, ids, n=60)
Block.forward = block_forward


## Optional. Research extension, induction heads

Trains a 2-layer attention-only model on repeated random tokens and prints the induction score and the ICL score every 50 steps. Watch for the step where both change together.

In [ ]:
# Research extension (optional). Induction score and ICL score on a 2-layer attention-only model
# trained FROM SCRATCH on random tokens that repeat with a period of 8 to 32. Same setup as induction.py
# (run lltGeneration-t1-induction-toy-L2-s0). Set LAYERS = 1 for the control, where no induction head can form.
LAYERS, V, D, H, N, STEPS = 2, 128, 64, 4, 32, 1500

class AttnBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1 = RMSNorm(D)
        self.qkv = nn.Linear(D, 3 * D, bias=False)
        self.o = nn.Linear(D, D, bias=False)

class AttnOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.blocks = nn.ModuleList([AttnBlock() for _ in range(LAYERS)])
        self.norm = RMSNorm(D)
        self.head = nn.Linear(D, V, bias=False)
        self.head.weight = self.emb.weight
        for n, p in self.named_parameters():
            if p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * LAYERS) if n.endswith("o.weight") else 0.02)
    def forward(self, idx, return_maps=False):
        B, T = idx.shape
        cos, sin = rope_tables(0, T, D // H)
        mask = torch.ones(T, T, dtype=torch.bool).tril()
        x, maps = self.emb(idx), []
        for b in self.blocks:
            q, k, v = (t.view(B, T, H, D // H).transpose(1, 2) for t in b.qkv(b.n1(x)).split(D, 2))
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
            att = ((q @ k.transpose(-1, -2)) / math.sqrt(D // H)).masked_fill(~mask, float("-inf")).softmax(-1)
            maps.append(att)
            x = x + b.o((att @ v).transpose(1, 2).reshape(B, T, D))
        logits = self.head(self.norm(x))
        return (logits, maps) if return_maps else logits

def train_batch(batch, gen):
    # n random tokens repeated to length 64, n drawn from 8 to 32 for each sequence.
    # The period varies, so a head that looks a fixed distance back cannot predict the repeats.
    x = torch.empty(batch, 2 * N, dtype=torch.long)
    for b in range(batch):
        n = int(torch.randint(8, 33, (1,), generator=gen))
        x[b] = torch.randint(0, V, (n,), generator=gen).repeat(2 * N // n + 1)[:2 * N]
    return x

def probe(seed):
    r = torch.randint(0, V, (64, N), generator=torch.Generator().manual_seed(seed))
    return torch.cat([r, r], 1)                            # 32 random tokens, then the same 32 again

@torch.no_grad()
def scores(m):
    _, maps = m(probe(1234), return_maps=True)             # the same probes as induction.py
    dst = torch.arange(N, 2 * N)                           # every token of the second copy
    ind = maps[-1][:, :, dst, dst - N + 1].mean((0, 2))    # attention to the token after its first occurrence
    x = probe(999)
    logits = m(x)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, V), x[:, 1:].reshape(-1), reduction="none").view(64, -1)
    icl = (loss[:, 59] - loss[:, 9]).mean().item()         # loss at token 60 minus loss at token 10
    return ind.max().item(), icl

torch.manual_seed(0)
toy = AttnOnly()
opt = torch.optim.AdamW(toy.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.0)
gen = torch.Generator().manual_seed(100)
for step in range(STEPS + 1):
    if step % 50 == 0:
        toy.eval(); ind, icl = scores(toy); toy.train()
        print(f"step {step:4d}  induction score {ind:.2f}  ICL score {icl:+.2f} nats")
    if step == STEPS:
        break
    for g in opt.param_groups:
        g["lr"] = 1e-3 * min(1.0, (step + 1) / 100)
    x = train_batch(32, gen)
    loss = F.cross_entropy(toy(x[:, :-1]).reshape(-1, V), x[:, 1:].reshape(-1))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
